# Hard Case: Pagination Kompleks

Banyak situs membagi data ke **banyak halaman**. Dua pola umum:

1. **Tombol "Next" berupa link biasa** → cukup `requests` + BeautifulSoup. Daripada
   menebak jumlah halaman, kita **ikuti tombol Next sampai hilang**.
2. **Tombol "Next" via JavaScript** (butuh klik, URL bisa tidak berubah) → pakai **Selenium**.

Latihan pakai `https://quotes.toscrape.com` (ada tombol Next di tiap halaman).

> Aturan main: kalau bisa dengan `requests` (cara 1), **jangan** buru-buru pakai Selenium.
> Selenium hanya untuk yang butuh JS/interaksi.

**Tooling:** `requests` + `beautifulsoup4` (cara 1), `selenium` + `WebDriverWait` (cara 2).


## Cara 1 — Ikuti tombol "Next" dengan `requests` (disarankan)

Polanya: scrape halaman → cari elemen Next (`<li class="next"><a href="...">`) →
kalau ada, lanjut ke URL itu; kalau tidak ada, **berhenti**. Pakai `urljoin` agar
URL relatif (`/page/2/`) jadi URL lengkap. Selalu beri **batas maksimum** halaman
sebagai pengaman dari loop tak berujung.


In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = "https://quotes.toscrape.com/"


def scrape_semua_halaman(start_url=BASE, max_halaman=20):
    url = start_url
    semua = []
    halaman = 0
    while url and halaman < max_halaman:  # max_halaman = pengaman
        halaman += 1
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        for q in soup.find_all("div", class_="quote"):
            semua.append(q.find("span", class_="text").get_text(strip=True))

        # cari tombol Next; kalau tidak ada -> url jadi None -> loop berhenti
        next_li = soup.find("li", class_="next")
        url = urljoin(BASE, next_li.find("a")["href"]) if next_li else None
        print(f"halaman {halaman:>2} -> total terkumpul: {len(semua)}")

    return semua


quotes = scrape_semua_halaman()
print("\nSelesai. Total quote:", len(quotes))


halaman  1 -> total terkumpul: 10


halaman  2 -> total terkumpul: 20


halaman  3 -> total terkumpul: 30


halaman  4 -> total terkumpul: 40


halaman  5 -> total terkumpul: 50


halaman  6 -> total terkumpul: 60


halaman  7 -> total terkumpul: 70


halaman  8 -> total terkumpul: 80


halaman  9 -> total terkumpul: 90


halaman 10 -> total terkumpul: 100

Selesai. Total quote: 100


## Cara 2 — Klik tombol "Next" dengan Selenium

Kalau tombol Next butuh JavaScript / klik (URL tidak selalu berubah), kita pakai Selenium.
Polanya sama: kumpulkan data → cari tombol Next → klik → ulangi sampai tombol Next hilang.

Pakai `WebDriverWait` untuk **menunggu** quote muncul dulu (jangan `time.sleep` asal),
dan `try/except NoSuchElementException` untuk mendeteksi halaman terakhir.


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException


def buat_driver(headless=True):
    o = Options()
    if headless:  # set False kalau mau lihat browsernya
        o.add_argument("--headless=new")
    o.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=o)


driver = buat_driver()
semua = []
try:
    driver.get("https://quotes.toscrape.com/js/")
    halaman = 0
    while halaman < 20:  # pengaman
        halaman += 1
        # tunggu quote (yang dirender JS) muncul
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".quote .text"))
        )
        for el in driver.find_elements(By.CSS_SELECTOR, ".quote .text"):
            semua.append(el.text)

        # cari tombol Next; tidak ada -> halaman terakhir
        try:
            driver.find_element(By.CSS_SELECTOR, "li.next a").click()
        except NoSuchElementException:
            break

    print(f"Selesai di halaman {halaman}, total {len(semua)} quote")
finally:
    driver.quit()


Selesai di halaman 10, total 100 quote


## Kesimpulan & Latihan

- **Ikuti tombol Next sampai hilang** lebih tahan banting daripada hardcode `range(1, N)`
  (kalau jumlah halaman berubah, kode tetap jalan).
- Selalu pasang **batas maksimum** halaman sebagai pengaman.
- Pilih `requests` dulu; Selenium hanya kalau Next butuh JS/klik.

**Latihan:** ubah cara 1 agar tiap quote menyimpan **teks + penulis** (dict), lalu hitung
berapa quote per penulis. Contoh jawaban di sel berikut.


In [3]:
# Contoh jawaban latihan
from collections import Counter


def scrape_dengan_author(start_url=BASE, max_halaman=20):
    url, out, h = start_url, [], 0
    while url and h < max_halaman:
        h += 1
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for q in soup.find_all("div", class_="quote"):
            out.append(
                {
                    "text": q.find("span", class_="text").get_text(strip=True),
                    "author": q.find("small", class_="author").get_text(strip=True),
                }
            )
        nx = soup.find("li", class_="next")
        url = urljoin(BASE, nx.find("a")["href"]) if nx else None
    return out


data = scrape_dengan_author()
print("Total item:", len(data))
print("Top 5 penulis:", Counter(d["author"] for d in data).most_common(5))


Total item: 100
Top 5 penulis: [('Albert Einstein', 10), ('J.K. Rowling', 9), ('Marilyn Monroe', 7), ('Dr. Seuss', 6), ('Mark Twain', 6)]
